##### Customers:
1. **Active Users**: Number of users that made a purchase in last month + before that
2. **Average purchase value**: On average how much a customer spents.
3. **Average purchases**: how many purchaes average user made

In [2]:
import polars as pl

orders = pl.read_csv(source = "../data/orders.csv",
                    schema_overrides = {"order_purchase_timestamp": pl.Datetime,
                                        "order_approved_at": pl.Datetime,
                                        "order_delivered_carrier_date": pl.Datetime,
                                        "order_delivered_customer_date": pl.Datetime,
                                        "order_estimated_delivery_date": pl.Datetime})

items = pl.read_csv("../data/order_items.csv",
                    schema_overrides = {"shipping_limit_date": pl.Datetime})

sales = items.join(orders,
          on = "order_id",
          how = "left")

In [17]:
(sales.group_by("customer_id").agg(pl.col("order_id").n_unique().alias("purchase_count"))
      .filter(pl.col("purchase_count")>1))

customer_id,purchase_count
str,u32


Customers makes only one order. There is no person to order second time. Thus creating a active user seems unfeaseable. Also it is obvious that average user make 1 purchase since every customer makes only 1 purchase.

In [24]:
items.columns

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

In [23]:
average_order_spending = items.group_by("order_id").agg(pl.col("price").sum()).get_column("price").mean()
average_order_spending

137.75407637889447

In [28]:
average_items_in_order = items.group_by("order_id").agg(pl.col("order_item_id").count()).get_column("order_item_id").mean()
average_items_in_order

1.1417306873695092

In [32]:
average_item_price = items.get_column("price").mean()
average_item_price

120.6537390146472